In [ ]:
%py
%pip install boto3
%pip install botocore

# PySpark script: Automate migration of eligible files from Purgo S3 landing folder to archive folder using s3_file_process_log
# Purpose: Move files with file_status='SUCCESS' from S3 landing to archive folders as per log table, using AWS credentials from Databricks secrets
# Author: Giang Nguyen
# Date: 2025-09-29
# Description: This script reads eligible file records from purgo_playground.s3_file_process_log, validates required fields and file types, retrieves AWS credentials from Databricks secret scope 'aws_keys', and moves each file from its S3 landing path to the archive path. Errors are logged to purgo_playground.wrk_aws_secret_error. No changes are made to the log table. All operations use Databricks and PySpark best practices, with robust error handling and schema validation.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks

# Import required PySpark functions and types # Built-in PySpark
from pyspark.sql.types import StructType, StructField, StringType, TimestampType  
from pyspark.sql.functions import col  

# Import boto3 and botocore for S3 operations # pip install boto3, pip install botocore
import boto3  
from botocore.exceptions import ClientError  

# Databricks utilities for secret management # Built-in Databricks
dbutils = globals().get('dbutils') if 'dbutils' in globals() else None

def get_aws_credentials():
    """
    Retrieves AWS credentials from Databricks secret scope 'aws_keys'.
    Returns:
        dict: {'access_key': str, 'secret_key': str}
    Raises:
        Exception: If secret is missing or invalid
    """
    if dbutils is None:
        raise Exception("Databricks dbutils not available for secret retrieval")
    try:
        access_key = dbutils.secrets.get(scope="aws_keys", key="access_key")
    except Exception:
        raise Exception("Databricks secret 'aws_keys/access_key' not found")
    try:
        secret_key = dbutils.secrets.get(scope="aws_keys", key="secret_key")
    except Exception:
        raise Exception("Databricks secret 'aws_keys/secret_key' not found")
    if not access_key or not secret_key:
        raise Exception("Invalid AWS credentials provided")
    return {"access_key": access_key, "secret_key": secret_key}

def get_s3_client(aws_creds):
    """
    Initializes and returns a boto3 S3 client using provided credentials.
    Args:
        aws_creds (dict): AWS credentials
    Returns:
        boto3.client: S3 client
    Raises:
        Exception: If credentials are invalid
    """
    try:
        s3 = boto3.client(
            "s3",
            aws_access_key_id=aws_creds["access_key"],
            aws_secret_access_key=aws_creds["secret_key"],
        )
        # Test connection
        s3.list_buckets()
        return s3
    except Exception:
        raise Exception("Invalid AWS credentials provided")

def parse_s3_path(s3_path):
    """
    Parses S3 URI into bucket and key.
    Args:
        s3_path (str): S3 URI (e.g., s3://bucket/path/to/file)
    Returns:
        tuple: (bucket, key)
    Raises:
        Exception: If path is invalid
    """
    if not s3_path or not s3_path.startswith("s3://"):
        raise Exception(f"S3 path not found: {s3_path}")
    parts = s3_path.replace("s3://", "").split("/", 1)
    if len(parts) != 2 or not parts[0] or not parts[1]:
        raise Exception(f"S3 path not found: {s3_path}")
    return parts[0], parts[1].rstrip("/")

def is_supported_file_type(file_name):
    """
    Checks if the file type is supported for migration.
    Args:
        file_name (str): File name
    Returns:
        bool: True if supported, False otherwise
    """
    if not file_name or not isinstance(file_name, str):
        return False
    supported = [".csv", ".json", ".dcm"]
    for ext in supported:
        if file_name.lower().endswith(ext):
            return True
    return False

def file_exists_in_s3(s3_client, bucket, key):
    """
    Checks if a file exists in S3.
    Args:
        s3_client (boto3.client): S3 client
        bucket (str): S3 bucket
        key (str): S3 key
    Returns:
        bool: True if exists, False otherwise
    """
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        if e.response['Error']['Code'] == "404":
            return False
        else:
            raise
    except Exception:
        return False

def move_s3_file(s3_client, src_bucket, src_key, dest_bucket, dest_key):
    """
    Moves a file from source S3 location to destination S3 location.
    Args:
        s3_client (boto3.client): S3 client
        src_bucket (str): Source bucket
        src_key (str): Source key
        dest_bucket (str): Destination bucket
        dest_key (str): Destination key
    Returns:
        None
    Raises:
        Exception: If move fails
    """
    try:
        s3_client.copy_object(
            Bucket=dest_bucket,
            Key=dest_key,
            CopySource={'Bucket': src_bucket, 'Key': src_key}
        )
        s3_client.delete_object(Bucket=src_bucket, Key=src_key)
    except Exception as e:
        raise Exception(f"Failed to move file from {src_bucket}/{src_key} to {dest_bucket}/{dest_key}: {str(e)}")

def log_error_to_table(error_message):
    """
    Logs error message to wrk_aws_secret_error table.
    Args:
        error_message (str): Error message
    Returns:
        None
    """
    error_df = spark.createDataFrame([{"error_message": error_message}], "error_message STRING")
    error_df.write.mode("append").saveAsTable("purgo_playground.wrk_aws_secret_error")

def validate_log_table_schema(df):
    """
    Validates the schema of the s3_file_process_log DataFrame.
    Args:
        df (DataFrame): DataFrame to validate
    Returns:
        None
    Raises:
        Exception: If schema does not match expected
    """
    expected_schema = StructType([
        StructField("file_name", StringType(), True),
        StructField("s3_vendor_path", StringType(), True),
        StructField("s3_landing_path", StringType(), True),
        StructField("s3_archive_path", StringType(), True),
        StructField("file_status", StringType(), True),
        StructField("file_processed_date", TimestampType(), True),
    ])
    actual_schema = df.schema
    if len(actual_schema) != len(expected_schema):
        raise Exception("Schema column count mismatch")
    for i, field in enumerate(expected_schema):
        if actual_schema[i].name != field.name:
            raise Exception(f"Column name mismatch: {actual_schema[i].name} != {field.name}")
        if type(actual_schema[i].dataType) != type(field.dataType):
            raise Exception(f"Data type mismatch for {field.name}")

def main():
    """
    Main function to automate migration of eligible files from S3 landing to archive folder.
    Reads eligible files from log table, validates, and moves files using AWS credentials from Databricks secrets.
    Logs errors to wrk_aws_secret_error table.
    """
    try:
        # Set Unity Catalog and schema
        spark.catalog.setCurrentCatalog("purgo_databricks")
        spark.catalog.setCurrentDatabase("purgo_playground")
        # Read log table
        log_df = spark.table("purgo_playground.s3_file_process_log")
        validate_log_table_schema(log_df)
        # Filter eligible files (file_status = 'SUCCESS')
        eligible_df = log_df.filter(col("file_status") == "SUCCESS")
        # Collect eligible records
        eligible_files = eligible_df.collect()
        if not eligible_files:
            # No eligible files, exit gracefully
            return
        # Get AWS credentials and S3 client
        aws_creds = get_aws_credentials()
        s3_client = get_s3_client(aws_creds)
        for row in eligible_files:
            try:
                # Validate required fields
                if not row.file_name or not isinstance(row.file_name, str):
                    raise Exception(f"Missing file_name for file {row.file_name}")
                if not row.s3_landing_path:
                    raise Exception(f"Missing s3_landing_path for file {row.file_name}")
                if not row.s3_archive_path:
                    raise Exception(f"Missing s3_archive_path for file {row.file_name}")
                if not row.file_status:
                    raise Exception(f"Missing file_status for file {row.file_name}")
                # Validate file type
                if not is_supported_file_type(row.file_name):
                    raise Exception(f"Unsupported file type: {row.file_name.split('.')[-1] if row.file_name else 'UNKNOWN'}")
                # Parse S3 paths
                src_bucket, src_key_prefix = parse_s3_path(row.s3_landing_path)
                dest_bucket, dest_key_prefix = parse_s3_path(row.s3_archive_path)
                # Compose full S3 keys
                src_key = src_key_prefix + "/" + row.file_name if not src_key_prefix.endswith(row.file_name) else src_key_prefix
                dest_key = dest_key_prefix + "/" + row.file_name if not dest_key_prefix.endswith(row.file_name) else dest_key_prefix
                # Check file exists in landing
                if not file_exists_in_s3(s3_client, src_bucket, src_key):
                    raise Exception(f"S3 path not found: {row.s3_landing_path}")
                # Check if file exists in archive
                if file_exists_in_s3(s3_client, dest_bucket, dest_key):
                    # Remove from landing, do not duplicate
                    s3_client.delete_object(Bucket=src_bucket, Key=src_key)
                else:
                    # Move file
                    move_s3_file(s3_client, src_bucket, src_key, dest_bucket, dest_key)
            except Exception as e:
                log_error_to_table(str(e))
    except Exception as e:
        log_error_to_table(str(e))

if __name__ == "__main__":
    main()

# spark.stop()  # Do not stop SparkSession in Databricks
